<a href="https://colab.research.google.com/github/sardorbek-0515/computer_vision/blob/master/menyu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
print('Menu Detector!')

Menu Detector!


In [15]:
# _______________________
# Import Libraries
# _______________________
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [16]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Defire Datasete Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING ={
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping  | class consolidation
    'cheesecake': 'dessert', # label grouping  | class consolidation
    'kebab': 'kebab',
    'osh': 'osh',
    'manti': 'manti'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'osh', 'manti']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 0 ~ 255
# 0.0 ~ 1.0
# 0.234
# H, W C => C, H, W | RGB = Red, Green, Blue
# Normalize
# pixer = (pixer - mean) / std

Dataset_path: /content/drive/MyDrive/food101_dataset
6
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'osh': 4, 'manti': 5}


In [18]:
# --------------------
# Custom Dataset Class
# --------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)
        label = self.labels[idx]
        # print('label', label)
        try:
            image = Image.open(img_path).convert('RGB') # 3 Channel
            if image.mode == 'P' or image.mode == "RGBA": # png | gif | RGBA
              image = image.convert("RGBA").convert("RGB")
            else:
              image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images)) #
        if self.transform:
            image = self.transform(image)
        return image, label


In [19]:
# --------------------
# Gather and Split Data
# --------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class) # /content/drive/MyDrive/food101_dataset/hamburger
    print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')): # .txt, .docx
            full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food101_dataset/hamburger/100057.jpg
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split] #1000 | 800 train_data | 200 val_data
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/osh
class_path: /content/drive/MyDrive/food101_dataset/manti
66


In [20]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [21]:
# Rasmlarni 32 tadan olib, aralashtirib modelga beradi
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) #thread | parallel loading for speed

# Tekshirish rasmlarini 32 tadan, aralashtirmasdan beradi
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [22]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1") # pretrained model | lightweight | CNN | 1000 class | million
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze

In [23]:
# GPU bo'lsa GPU, bo'lmasa CPU tanlaydi
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

# Modelni tanlangan qurilmaga o'tkazadi
model = model.to(device)

device cuda


In [24]:
# Model xatosini hisoblaydi
criterion = nn.CrossEntropyLoss() # Loss Function | '70%' burger, '30%' osh

# Model vaznlarini yangilaydi, o'rganish tezligi 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001)  # weight

# NVIDIA GPU ishlatilganda tezlikni optimallashtiradi
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10% - 20%

In [25]:
# -----------
# Training Loop
# -----------

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
  model.train() # train mode
  running_loss = 0.0 # 70% | 30% Loss |100

  # We will print progress every 10 steps to show active training
  print_every = 10

  for batch_idx, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad() # zero the gradient
    outputs = model(images) # Forward Pass
    loss = criterion(outputs, labels) # Calculate Loss
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

    if (batch_idx + 1) % print_every == 0 or (batch_idx + 1) == len(train_loader):
      print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Step [{batch_idx+1}/{len(train_loader)}] | Loss: {running_loss/(batch_idx+1):.4f}")

  # Validation
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

  val_acc = 100 * correct / total # Calculate Validation Accuracy
  print(f'=== Epoch [{epoch+1}/{NUM_EPOCHS}] Completed | Val Accuracy: {val_acc:.2f}% ===\n')

  if val_acc > best_accuracy:
    best_accuracy = val_acc
    torch.save(model.state_dict(), '/content/menu_detector.pth')
    print("Saved new best model!")


Epoch [1/10] | Step [3/3] | Loss: 0.8206
=== Epoch [1/10] Completed | Val Accuracy: 100.00% ===

Saved new best model!
Epoch [2/10] | Step [3/3] | Loss: 0.0525
=== Epoch [2/10] Completed | Val Accuracy: 100.00% ===

Epoch [3/10] | Step [3/3] | Loss: 0.0105
=== Epoch [3/10] Completed | Val Accuracy: 100.00% ===

Epoch [4/10] | Step [3/3] | Loss: 0.0009
=== Epoch [4/10] Completed | Val Accuracy: 100.00% ===

Epoch [5/10] | Step [3/3] | Loss: 0.0003
=== Epoch [5/10] Completed | Val Accuracy: 100.00% ===

Epoch [6/10] | Step [3/3] | Loss: 0.0001
=== Epoch [6/10] Completed | Val Accuracy: 100.00% ===

Epoch [7/10] | Step [3/3] | Loss: 0.0001
=== Epoch [7/10] Completed | Val Accuracy: 100.00% ===

Epoch [8/10] | Step [3/3] | Loss: 0.0000
=== Epoch [8/10] Completed | Val Accuracy: 100.00% ===

Epoch [9/10] | Step [3/3] | Loss: 0.0000
=== Epoch [9/10] Completed | Val Accuracy: 100.00% ===

Epoch [10/10] | Step [3/3] | Loss: 0.0000
=== Epoch [10/10] Completed | Val Accuracy: 100.00% ===

